## Script de validación de modelo
### Se obtiene predicción de precio con data de junio como data de validación, usando el mejor modelo obtenido en TFM_Airbnb.ipynb

Para este script son necesarios los siguientes ficheros:


1.   requirements.txt
2.   FuncionesTFM.py
3.   Pipeline_limpieza_data.py
4.   geo_barcelona.geojson
5.   modelo_xgb.json
6.   airbnb_junio.csv

Nota: Este script fue realizado en Colab por lo que el path de lectura de ficheros incia en  "/content/", se recomienda verificar el path de ubicación si se trabaja en local.


In [1]:
# Instalar requirements con su versión
!pip install -r "/content/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.2 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=0c31dcbcc1c5c839e7f33b80e395c7c85a21b43f6cf43898330e7b9021faf79d
  Stored in di

In [13]:
# Librerias
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import sys
import re
import patsy
import ast
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import torch
import torch.nn as nn
from relativeImp import relativeImp
from  ydata_profiling import ProfileReport
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import chi2_contingency
import geopandas as gpd
import pickle
import sklearn.impute as skl_imp
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, mean_absolute_error, median_absolute_error, f1_score, roc_auc_score, roc_curve



In [3]:
# Funciones
execfile("/content/FuncionesTFM.py")
execfile("/content/Pipeline_limpieza_data.py")
execfile("/content/geo_barcelona.geojson")

In [4]:
# Data a procesar para predecir
data = pd.read_csv('/content/airbnb_junio.csv')

In [5]:
#Obtener la data procesada
geo = gpd.read_file("geo_barcelona.geojson").to_crs(25831)
data_limpia = preparar_limpieza(data)

In [6]:
# Leo  nuestro modelo json
from xgboost import XGBRegressor

model = XGBRegressor()
model.load_model("/content/modelo_xgb.json")


In [7]:
# SELECCIÓN DE VARIABLES
variables_modelo = list(model.feature_names_in_)
X_final = data_limpia[variables_modelo].astype(float)

#  PREDICCIÓN
pred = model.predict(X_final)

In [8]:
prediccion = pd.DataFrame(pred)
prediccion.columns = ['pred_precio_log']

In [9]:
prediccion.head()

,pred_precio_log
0,5.954765
1,5.951742
2,6.017750
3,3.621529
4,5.251431


In [10]:

prediccion['precio_pred'] = np.exp(prediccion['pred_precio_log'])
prediccion['precio_pred'] = prediccion['precio_pred'].round(2)

In [11]:
prediccion

,pred_precio_log,precio_pred
0,5.954765,385.589996
1,5.951742,384.420013
2,6.017750,410.649994
3,3.621529,37.389999
4,5.251431,190.839996
...,...,...
13350,4.245271,69.769997
13351,4.245271,69.769997
13352,5.893930,362.829987
13353,5.902063,365.790009


In [14]:
# Evaluar la predicción con los datos de junio
y_junio = data_limpia['price_log']
r2_junio = r2_score(y_junio, prediccion['pred_precio_log'])
rmse_junio = np.sqrt(mean_squared_error(y_junio, prediccion['pred_precio_log']))
y_junio_euros = np.exp(y_junio)
pred_junio_euros = np.exp(prediccion['pred_precio_log'])
mae_junio = mean_absolute_error(y_junio_euros, pred_junio_euros)
mediana_junio = median_absolute_error(y_junio_euros, pred_junio_euros)

print(f"R² junio: {r2_junio:.4f}")
print(f"RMSE junio: {rmse_junio:.4f}")
print(f"MAE junio: {mae_junio:.2f}")
print(f"Mediana del error junio: {mediana_junio:.2f}")

R² junio: 0.8747
RMSE junio: 0.3082
MAE junio: 56.82
Mediana del error junio: 23.97


In [15]:
# Guardamos  la prediccion junio
prediccion = prediccion.join(data_limpia[['id']])
prediccion.to_csv('prediccion_junio.csv', index=False)
prediccion

,pred_precio_log,precio_pred,id
0,5.954765,385.589996,18674
1,5.951742,384.420013,23197
2,6.017750,410.649994,34981
3,3.621529,37.389999,36763
4,5.251431,190.839996,40983
...,...,...,...
13350,4.245271,69.769997,1714453367636295798
13351,4.245271,69.769997,1714507108958061169
13352,5.893930,362.829987,1714524223290226460
13353,5.902063,365.790009,1714528494413566181
